# 03. Intent Classification Graph 테스트

**목적**: `app/graphs/intent_graph.py`의 의도 분류 + RAG 파이프라인 동작 확인

**그래프 흐름**
```
START → intent_classifier → retriever_call → verify_retrieval
                                                    ↓
                              proceed_to_llm → llm_call → END
                              retry_retrieval → retriever_call (재시도)
                              proceed_without_docs → llm_call_without_docs → END
```

**체크리스트**
- [ ] 그래프 구조 시각화
- [ ] 관련 문서가 있는 질문 → llm_call 경로
- [ ] 관련 문서가 없는 질문 → llm_call_without_docs 경로
- [ ] 의도 분류 결과 확인
- [ ] llm_calls 카운트 확인

In [ ]:
import sys
sys.path.insert(0, '..')

## 1. 그래프 구축

In [ ]:
from app.retriever import build_vectorstore
from app.graphs.intent_graph import build_intent_graph

documents = [
    "인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습하고 추론할 수 있도록 하는 기술입니다.",
    "머신러닝은 AI의 한 분야로, 데이터로부터 패턴을 학습하여 예측이나 분류를 수행합니다.",
    "딥러닝은 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야입니다.",
    "자연어처리(NLP)는 인간의 언어를 컴퓨터가 이해하고 처리할 수 있도록 하는 AI 기술입니다.",
    "컴퓨터 비전은 이미지나 비디오에서 의미 있는 정보를 추출하는 AI 분야입니다.",
]

vectorstore = build_vectorstore(documents)
graph = build_intent_graph(vectorstore)
print("그래프 구축 완료")

## 2. 그래프 구조 시각화

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

## 3. 관련 문서 있는 질문 테스트

In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "컴퓨터 비전에 대해 알려줘"}]})

print(f"분류된 의도  : {result['intent']}")
print(f"LLM 호출 수  : {result['llm_calls']}")
print(f"재시도 횟수  : {result['retrieval_attempts']}")
print(f"\n최종 답변:\n{result['messages'][-1].content}")

## 4. 관련 문서 없는 질문 테스트 (fallback 경로)

In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "오늘 날씨가 좋아요"}]})

print(f"분류된 의도  : {result['intent']}")
print(f"LLM 호출 수  : {result['llm_calls']}")
print(f"재시도 횟수  : {result['retrieval_attempts']}")
print(f"\n최종 답변:\n{result['messages'][-1].content}")

## 5. 의도별 분류 결과 확인

In [ ]:
test_cases = [
    "딥러닝이 뭐야?",                          # 개념정의 예상
    "머신러닝과 딥러닝의 차이가 뭐야?",         # 비교분석 예상
    "NLP는 어떻게 작동해?",                    # 기술질문 예상
]

for query in test_cases:
    result = graph.invoke({"messages": [{"role": "user", "content": query}]})
    print(f"질문: {query}")
    print(f"의도: {result['intent']}\n")